# DuckDB Elite Football Tracking Analysis

Titled query blocks for curated denormalized tracking outputs. This notebook adds off-ball support analysis, a coarse pitch-control proxy, and common elite-club workflow checks.

- Update `curated_dir` to the curated Parquet folder you want to scan.
- `pitch_length` and `pitch_width` are expected to come from each match metadata file when available.
- Remote storage access in the pipeline can use per-profile `storage_options` in `config.yaml`.


## 0. Dataset Coverage and View Setup

Create a DuckDB view over the curated Parquet files and confirm the match range before running the analysis sections below.


In [ ]:
from pathlib import Path

import duckdb

curated_dir = Path(r"C:\Users\adamm\Downloads\tracking_files\tracking\curated")
sample_limit = 200

con = duckdb.connect()
con.execute(
    f"""
    CREATE OR REPLACE VIEW tracking AS
    SELECT *
    FROM read_parquet('{curated_dir.as_posix()}/*.parquet')
    """
)
con.execute(
    """
    SELECT COUNT(*) AS rows, MIN(opta_match_id) AS min_match, MAX(opta_match_id) AS max_match
    FROM tracking
    """
).df()


## 1. Nearest Player to Ball by Frame

A simple possession-pressure view: the closest player to the ball for each team and frame.


In [ ]:
con.execute(
    f"""
    SELECT
        fixture,
        match_date,
        team,
        period,
        frame_id,
        MIN(player_ball_distance) AS nearest_player_to_ball_m
    FROM tracking
    GROUP BY 1,2,3,4,5
    ORDER BY match_date, fixture, period, frame_id, team
    LIMIT {sample_limit}
    """
).df()


## 2. Team Width, Depth, and Centroid

A common shape summary used to understand spacing and the basic footprint of each team over time.


In [ ]:
con.execute(
    f"""
    SELECT
        fixture,
        match_date,
        team,
        period,
        frame_bucket,
        MAX(player_x) - MIN(player_x) AS team_width_m,
        MAX(player_y) - MIN(player_y) AS team_depth_m,
        AVG(player_x) AS team_centroid_x_m,
        AVG(player_y) AS team_centroid_y_m
    FROM tracking
    GROUP BY 1,2,3,4,5
    ORDER BY match_date, fixture, period, frame_bucket, team
    LIMIT {sample_limit}
    """
).df()


## 3. Inter-Team Compactness

Distance between team centroids by frame. This is a quick proxy for game stretch and compactness.


In [ ]:
con.execute(
    f"""
    WITH frame_team_shapes AS (
        SELECT
            opta_match_id,
            fixture,
            match_date,
            period,
            frame_id,
            team,
            AVG(player_x) AS centroid_x,
            AVG(player_y) AS centroid_y
        FROM tracking
        GROUP BY 1,2,3,4,5,6
    ),
    paired AS (
        SELECT
            home.fixture,
            home.match_date,
            home.period,
            home.frame_id,
            SQRT(
                POWER(home.centroid_x - away.centroid_x, 2) +
                POWER(home.centroid_y - away.centroid_y, 2)
            ) AS centroid_distance_m
        FROM frame_team_shapes home
        JOIN frame_team_shapes away
          ON home.opta_match_id = away.opta_match_id
         AND home.period = away.period
         AND home.frame_id = away.frame_id
         AND home.team = 'home'
         AND away.team = 'away'
    )
    SELECT *
    FROM paired
    ORDER BY match_date, fixture, period, frame_id
    LIMIT {sample_limit}
    """
).df()


## 4. Ball-Zone Occupancy and Support Density

Summarises where the ball lives and how much nearby support each team tends to have in those zones.


In [ ]:
con.execute(
    f"""
    SELECT
        fixture,
        match_date,
        team,
        ball_zone,
        COUNT(*) AS player_rows,
        AVG(player_ball_distance) AS avg_player_ball_distance_m,
        SUM(CASE WHEN has_ball_possession THEN 1 ELSE 0 END) AS players_in_possession_radius
    FROM tracking
    GROUP BY 1,2,3,4
    ORDER BY match_date, fixture, team, player_rows DESC
    LIMIT {sample_limit}
    """
).df()


## 5. Player Support Profiles

Profiles how often individual players operate close to the ball and how much speed they carry in those phases.


In [ ]:
con.execute(
    f"""
    SELECT
        fixture,
        match_date,
        player_name,
        player_position,
        AVG(player_ball_distance) AS avg_distance_to_ball_m,
        AVG(player_speed) AS avg_player_speed_mps,
        SUM(CASE WHEN player_ball_distance <= 5 THEN 1 ELSE 0 END) AS frames_within_5m,
        SUM(CASE WHEN player_ball_distance <= 10 THEN 1 ELSE 0 END) AS frames_within_10m
    FROM tracking
    GROUP BY 1,2,3,4
    ORDER BY match_date, fixture, avg_distance_to_ball_m ASC
    LIMIT {sample_limit}
    """
).df()


## 6. Off-Ball Support and Lane Occupancy

A lightweight off-ball view for the team in possession. It highlights non-ball-carriers, their speed, and whether they stay in the same or adjacent lane to the ball.


In [ ]:
con.execute(
    f"""
    WITH possession_frames AS (
        SELECT
            opta_match_id,
            fixture,
            match_date,
            period,
            frame_id,
            MAX(CASE WHEN last_touch IN ('home', 'away') THEN last_touch END) AS possession_team
        FROM tracking
        GROUP BY 1,2,3,4,5
    ),
    off_ball AS (
        SELECT
            t.fixture,
            t.match_date,
            t.team,
            t.player_name,
            t.player_position,
            COUNT(*) AS off_ball_frames,
            AVG(t.player_speed) AS avg_off_ball_speed_mps,
            AVG(t.player_ball_distance) AS avg_distance_to_ball_m,
            AVG(ABS(t.player_y - t.ball_y)) AS avg_lane_offset_m,
            SUM(CASE WHEN ABS(t.player_y - t.ball_y) <= 8 THEN 1 ELSE 0 END) AS same_lane_frames,
            SUM(CASE WHEN ABS(t.player_y - t.ball_y) > 8 AND ABS(t.player_y - t.ball_y) <= 18 THEN 1 ELSE 0 END) AS adjacent_lane_frames
        FROM tracking t
        JOIN possession_frames p
          ON t.opta_match_id = p.opta_match_id
         AND t.period = p.period
         AND t.frame_id = p.frame_id
        WHERE p.possession_team IN ('home', 'away')
          AND t.team = p.possession_team
          AND NOT t.has_ball_possession
          AND t.player_name IS NOT NULL
        GROUP BY 1,2,3,4,5
    )
    SELECT *
    FROM off_ball
    ORDER BY match_date, fixture, off_ball_frames DESC, avg_off_ball_speed_mps DESC
    LIMIT {sample_limit}
    """
).df()


## 7. Pitch Control Proxy by Coarse Grid

This is a coarse proxy, not a full pitch-control model. It assigns each grid cell to the team with the nearest player and estimates frame-level territorial control.


In [ ]:
con.execute(
    f"""
    WITH frame_pitch AS (
        SELECT
            opta_match_id,
            fixture,
            match_date,
            period,
            frame_id,
            MAX(pitch_length) AS pitch_length,
            MAX(pitch_width) AS pitch_width
        FROM tracking
        GROUP BY 1,2,3,4,5
    ),
    grid AS (
        SELECT *
        FROM (
            VALUES
                (-0.40, -0.30), (-0.40, 0.00), (-0.40, 0.30),
                (-0.20, -0.30), (-0.20, 0.00), (-0.20, 0.30),
                (0.00, -0.30), (0.00, 0.00), (0.00, 0.30),
                (0.20, -0.30), (0.20, 0.00), (0.20, 0.30),
                (0.40, -0.30), (0.40, 0.00), (0.40, 0.30)
        ) AS grid(x_factor, y_factor)
    ),
    distances AS (
        SELECT
            fp.fixture,
            fp.match_date,
            fp.period,
            fp.frame_id,
            g.x_factor,
            g.y_factor,
            MIN(
                CASE WHEN t.team = 'home' THEN
                    SQRT(
                        POWER(t.player_x - (g.x_factor * fp.pitch_length / 2.0), 2) +
                        POWER(t.player_y - (g.y_factor * fp.pitch_width / 2.0), 2)
                    )
                END
            ) AS home_min_distance_m,
            MIN(
                CASE WHEN t.team = 'away' THEN
                    SQRT(
                        POWER(t.player_x - (g.x_factor * fp.pitch_length / 2.0), 2) +
                        POWER(t.player_y - (g.y_factor * fp.pitch_width / 2.0), 2)
                    )
                END
            ) AS away_min_distance_m
        FROM frame_pitch fp
        CROSS JOIN grid g
        JOIN tracking t
          ON t.opta_match_id = fp.opta_match_id
         AND t.period = fp.period
         AND t.frame_id = fp.frame_id
        GROUP BY 1,2,3,4,5,6
    ),
    control AS (
        SELECT
            fixture,
            match_date,
            period,
            frame_id,
            SUM(CASE WHEN home_min_distance_m < away_min_distance_m THEN 1 ELSE 0 END) AS home_controlled_cells,
            SUM(CASE WHEN away_min_distance_m < home_min_distance_m THEN 1 ELSE 0 END) AS away_controlled_cells,
            AVG(home_min_distance_m - away_min_distance_m) AS avg_home_advantage_m
        FROM distances
        GROUP BY 1,2,3,4
    )
    SELECT *
    FROM control
    ORDER BY match_date, fixture, period, frame_id
    LIMIT {sample_limit}
    """
).df()


## 8. Common Elite-Club Checks: Overloads and Counterpress

A compact frame-level table for ball-side overloads, second-ring support, and nearby opposition pressure. These are common starting points in elite-club tracking workflows.


In [ ]:
con.execute(
    f"""
    WITH frame_team AS (
        SELECT
            opta_match_id,
            fixture,
            match_date,
            period,
            frame_id,
            team,
            MAX(last_touch) AS last_touch,
            SUM(CASE WHEN player_ball_distance <= 10 THEN 1 ELSE 0 END) AS players_within_10m_of_ball,
            SUM(CASE WHEN player_ball_distance > 10 AND player_ball_distance <= 25 THEN 1 ELSE 0 END) AS players_in_second_ring,
            MAX(player_x) - MIN(player_x) AS team_width_m,
            MAX(player_y) - MIN(player_y) AS team_depth_m,
            AVG(player_speed) AS avg_team_speed_mps
        FROM tracking
        GROUP BY 1,2,3,4,5,6
    ),
    paired AS (
        SELECT
            pos.fixture,
            pos.match_date,
            pos.period,
            pos.frame_id,
            pos.team AS possession_team,
            pos.players_within_10m_of_ball AS support_near_ball,
            pos.players_in_second_ring AS support_second_ring,
            def.players_within_10m_of_ball AS counterpress_near_ball,
            def.players_in_second_ring AS defending_second_ring,
            pos.players_within_10m_of_ball - def.players_within_10m_of_ball AS overload_delta_10m,
            pos.team_width_m AS possession_team_width_m,
            pos.team_depth_m AS possession_team_depth_m,
            def.team_width_m AS defending_team_width_m,
            def.team_depth_m AS defending_team_depth_m,
            pos.avg_team_speed_mps AS possession_avg_speed_mps,
            def.avg_team_speed_mps AS defending_avg_speed_mps
        FROM frame_team pos
        JOIN frame_team def
          ON pos.opta_match_id = def.opta_match_id
         AND pos.period = def.period
         AND pos.frame_id = def.frame_id
        WHERE pos.team = pos.last_touch
          AND def.team <> pos.team
    )
    SELECT *
    FROM paired
    ORDER BY match_date, fixture, period, frame_id
    LIMIT {sample_limit}
    """
).df()
